In [9]:
import pandas as pd
import numpy as np
import os

# 1. Load the original parquet to match the exact dates and index format
df_orig = pd.read_parquet('../../data/01_raw/returns.parquet')
df_orig2 = pd.read_parquet('../../data/02_clean/returns_2010_2019.parquet')
dates = df_orig.index.union(df_orig2.index)

n_stocks = 401
n_lead = 50
n_lag = 50
n_noise = 301

# Typical daily volatility (e.g. 2% daily moves)
sigma = 0.02 
np.random.seed(42)

# 2. Generate random returns for the 50 LEADING stocks
lead_returns = np.random.normal(0, sigma, size=(len(dates), n_lead))
df_lead = pd.DataFrame(
    lead_returns, 
    index=dates, 
    columns=[f"LEAD_{i+1:02d}" for i in range(n_lead)]
)

# 3. Generate returns for the 50 LAGGING stocks
# We use a linear combination: F(t) = rho * L(t-1) + sqrt(1 - rho^2) * noise
# This preserves the overall variance (sigma^2) of the returns while ensuring a strong d+1 correlation.
rho = 0.8
lag_returns = np.zeros_like(lead_returns)
lag_returns[0] = np.random.normal(0, sigma, size=n_lead) # First day has no d-1 to follow
lag_returns[1:] = (
    rho * lead_returns[:-1] + 
    np.sqrt(1 - rho**2) * np.random.normal(0, sigma, size=(len(dates)-1, n_lead))
)

df_lag = pd.DataFrame(
    lag_returns, 
    index=dates, 
    columns=[f"LAG_{i+1:02d}" for i in range(n_lag)]
)

# 4. Generate 301 UNRELATED stocks with random noise
noise_returns = np.random.normal(0, sigma, size=(len(dates), n_noise))
df_noise = pd.DataFrame(
    noise_returns, 
    index=dates, 
    columns=[f"NOISE_{i+1:03d}" for i in range(n_noise)]
)

# 5. Combine everything into a single DataFrame (401 columns)
df_synthetic = pd.concat([df_lead, df_lag, df_noise], axis=1)
df_synthetic.index.name = df_orig.index.name

# 6. Save the synthetic data
out_path = '../../data/01_raw/synthetic_returns.parquet'
df_synthetic.to_parquet(out_path)

print(f"Synthetic data successfully generated! Saved to: {out_path}")
print(f"Shape: {df_synthetic.shape}")
display(df_synthetic.head())

Synthetic data successfully generated! Saved to: ../../data/01_raw/synthetic_returns.parquet
Shape: (3772, 401)


,LEAD_01,LEAD_02,LEAD_03,LEAD_04,LEAD_05,LEAD_06,LEAD_07,LEAD_08,LEAD_09,LEAD_10,...,NOISE_292,NOISE_293,NOISE_294,NOISE_295,NOISE_296,NOISE_297,NOISE_298,NOISE_299,NOISE_300,NOISE_301
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-05,0.009934,-0.002765,0.012954,0.030461,-0.004683,-0.004683,0.031584,0.015349,-0.009389,0.010851,...,-0.001996,-0.013986,-0.009462,-0.039516,0.014158,-0.002226,0.023230,-0.004758,-0.020768,0.017810
2010-01-06,0.006482,-0.007702,-0.013538,0.012234,0.020620,0.018626,-0.016784,-0.006184,0.006625,0.019511,...,-0.008101,-0.018737,-0.014334,-0.015720,-0.027808,0.019536,-0.013213,-0.001137,-0.005995,-0.023415
2010-01-07,-0.028307,-0.008413,-0.006854,-0.016046,-0.003226,0.008081,0.037724,0.003492,0.005151,-0.001489,...,-0.011226,-0.022923,0.002367,0.005148,-0.008401,0.037257,0.026293,-0.004621,-0.032559,-0.016139
2010-01-08,0.005010,0.006929,-0.013600,0.004645,0.005861,-0.014287,0.037315,0.009477,-0.023826,0.013131,...,-0.019807,0.037261,0.018781,-0.005226,-0.008193,0.007708,0.001213,-0.016952,0.015368,-0.002802
2010-01-11,0.007156,0.011216,0.021661,0.021076,-0.027553,-0.018757,0.010301,0.010276,0.010301,0.077055,...,0.024003,0.016814,-0.000887,-0.007092,-0.048741,0.028680,-0.044846,0.037554,-0.020806,0.009549
